# 02 May 05 Linear Clip — LR & LR decay search

Фиксируем **LinearBidder bin clip** (как в старом «fixed baseline»), а через Optuna перебираем **скидку MDP для DQN (`dqn_gamma`)**, **стартовые lr** для DQN и RewardNet и **стратегии затухания learning rate**:

- **DQN `dqn_gamma`**: сетка **1.0, 0.999, 0.99** (дисконт по горизонту Q-learning).
- **DQN LR decay**: `ExponentialLR` (несколько множителей по LR), либо без scheduler, либо `CosineAnnealingWarmRestarts` с периодами 2000 / 8000 шагов обучения (один `scheduler.step` на один gradient step).
- **RewardNet**: без decay или `ExponentialLR` с $\gamma \in \{0.9995, 0.9999\}$.

Профиль: `may06_default_linear_clip_lr_scheduler_search` в `profiles.py` (`n_trials=36`). Число триалов можно переопределить константой ниже.

In [1]:
import sys
import pickle
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params

{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [3]:
RUN_NAME = 'may06_clip_lr_scheduler_search'
DRLB_PROFILE = 'may05_default_clip_lr_scheduler_search'
VERBOSE = False
SHOW_PROGRESS = True
# None = взять n_trials из профиля (36)
N_TRIALS_OVERRIDE = None

In [4]:
RUN_NAME = "may06_default_clip_layer_norm_lr_scheduler_search_10"
DRLB_PROFILE = "may05_default_clip_lr_scheduler_search"
VERBOSE = False

config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set="full_train_val_holdout")
config = replace(config, n_trials=10, refit_on="train_plus_val", max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data["base_drlb_params"])
reference_model_params = dict(profile_data["reference_model_params"])

# чтобы сохранить условия из may_06/03:
base_drlb_params["dqn_layer_norm"] = True
base_drlb_params["bid_lower_clip"] = 3
base_drlb_params["bid_upper_clip"] = 8
base_drlb_params["dqn_layer_norm"] = True # Вот тут добавили LN, криво но все же


base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
tuning = summary['tuning']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'n_trials': tuning['n_trials'],
    'study_best_val_clicks': tuning['study_best_value'],
    'best_trial': tuning['best_trial_number'],
    'best_params': tuning['best_params'],
    'best_val_metrics': tuning['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}

autobidder_check campaigns: 100%|██████████| 257/257 [00:17<00:00, 14.94campaign/s, campaign_id=7.46e+7]
[I 2026-05-06 09:49:04,686] A new study created in memory with name: no-name-8dd0dce3-c913-4830-9c39-f81b405c3b81
Best trial: 0. Best value: 2275.17:  10%|█         | 1/10 [04:15<38:21, 255.73s/it]

[I 2026-05-06 09:53:20,412] Trial 0 finished with value: 2275.1695432341603 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.003, 'reward_net_lr': 0.01, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'none'}. Best is trial 0 with value: 2275.1695432341603.


Best trial: 0. Best value: 2275.17:  20%|██        | 2/10 [08:28<33:53, 254.18s/it]

[I 2026-05-06 09:57:33,505] Trial 1 finished with value: 2132.413936153344 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.0001, 'reward_net_lr': 0.003, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'exp_0.9995'}. Best is trial 0 with value: 2275.1695432341603.


Best trial: 0. Best value: 2275.17:  30%|███       | 3/10 [12:49<30:00, 257.16s/it]

...

[I 2026-05-06 10:25:01,847] Trial 8 finished with value: 2063.3383983175286 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.0003, 'reward_net_lr': 0.001, 'dqn_lr_decay': 'cosine_T2000', 'reward_net_lr_decay': 'exp_0.9999'}. Best is trial 0 with value: 2275.1695432341603.


Best trial: 0. Best value: 2275.17: 100%|██████████| 10/10 [39:37<00:00, 237.75s/it]


[I 2026-05-06 10:28:42,191] Trial 9 finished with value: 2133.9415959600983 and parameters: {'dqn_gamma': 0.999, 'dqn_lr': 0.0001, 'reward_net_lr': 0.0003, 'dqn_lr_decay': 'exp_0.999', 'reward_net_lr_decay': 'exp_0.9995'}. Best is trial 0 with value: 2275.1695432341603.


autobidder_check campaigns: 100%|██████████| 257/257 [00:17<00:00, 14.73campaign/s, campaign_id=7.46e+7]


{'run_name': 'may06_default_clip_layer_norm_lr_scheduler_search_10',
 'profile': 'may05_default_clip_lr_scheduler_search',
 'n_trials': 10,
 'study_best_val_clicks': 2275.1695432341603,
 'best_trial': 0,
 'best_params': {'dqn_gamma': 1.0,
  'dqn_lr': 0.003,
  'reward_net_lr': 0.01,
  'dqn_lr_decay': 'cosine_T8000',
  'reward_net_lr_decay': 'none'},
 'best_val_metrics': {'cpc_relative': 1246.1096500550593,
  'rmse': 1.2665033470742137,
  'clicks_sum': 2275.1695432341603,
  'quickspend': 0.03501945525291829,
  'skipped_campaigns': 0,
  'time_inference_sec': 17.11565113067627,
  'time_overall_sec': 21.43613600730896,
  'average_end_balance_share': 0.7576326868809378,
  'label': 'best_val',
  'train_steps': 48240,
  'last_dqn_loss': 55.4828987121582,
  'last_reward_net_loss': 444.15728759765625,
  'dqn_loss_mean': 70.7272646382001,
  'dqn_loss_p95': 204.35160827636722,
  'reward_net_loss_mean': 564.8580795537238,
  'reward_net_loss_p95': 1893.207421875,
  'reward_signal_mean': 9.3249567229

In [5]:
trials_df = pd.DataFrame(tuning['all_trials_summary'])
if 'clicks_sum' in trials_df.columns:
    trials_df = trials_df.sort_values('clicks_sum', ascending=False)
trials_df

,trial,dqn_gamma,dqn_lr,reward_net_lr,dqn_lr_decay,reward_net_lr_decay,rmse,clicks_sum,cpc_relative,quickspend,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,reward_net_loss_mean
0,0,1.000,0.0030,0.0100,cosine_T8000,none,1.266503,2275.169543,1246.109650,0.035019,55.482899,444.157288,70.727265,564.858080
7,7,1.000,0.0010,0.0003,cosine_T8000,none,1.252296,2271.408373,1090.261749,0.019455,24.296581,204.018768,46.429298,232.770289
4,4,0.999,0.0001,0.0003,none,exp_0.9999,1.467804,2164.693146,584.724058,0.058366,164.795532,797.796265,25.015321,289.814781
6,6,0.999,0.0001,0.0010,cosine_T2000,exp_0.9999,1.505457,2156.305346,429.175130,0.058366,79.532990,264.587646,16.782638,248.506679
9,9,0.999,0.0001,0.0003,exp_0.999,exp_0.9995,1.585126,2133.941596,312.604187,0.046693,6.339367,221.570618,5.430953,357.980175
1,1,1.000,0.0001,0.0030,cosine_T8000,exp_0.9995,1.599759,2132.413936,312.627115,0.058366,67.211548,674.136963,20.195052,314.129637
3,3,1.000,0.0001,0.0030,cosine_T8000,exp_0.9995,1.599759,2132.413936,312.627115,0.058366,67.211548,674.136963,20.195052,314.129637
8,8,1.000,0.0003,0.0010,cosine_T2000,exp_0.9999,1.288594,2063.338398,856.498051,0.027237,85.512108,237.195953,36.732944,245.673615
5,5,1.000,0.0010,0.0100,exp_0.9999,exp_0.9995,1.258288,1717.061824,2101.450201,0.019455,124.697525,126.829239,65.449865,190.172326
2,2,0.990,0.0010,0.0010,exp_0.9999,exp_0.9999,1.949026,1588.093302,195.625356,0.062257,10.722288,161.995453,13.243161,154.551152


In [6]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])

,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
